In [14]:
import pandas as pd
import re
from tqdm import tqdm

In [25]:
df = pd.read_csv("Outputs/qwen3_ut_zs.csv")

df.head()

,sa,hi,skr_out,syllable_count,skr_meter,meter_cd,comments,prompt,split,thinking,model_out
0,उक्तोऽयमर्थ आचार्यों मया कर्णेन चासकृत् पुनरेव...,आचार्यो ! मैंने और कर्णने यह बात आपलोगों से कई...,"anuṣṭubh (1,2: pathyā; 3,4: pathyā)",32,Anuṣṭubh,Anuṣṭubh,NaN,Objective:\n\nGenerate a sanskrit verse in Anu...,test,NaN,आचार्य तव वचनं बहु विनिपातं विनिपातं च विनिपात...
1,अर्धकीलं च तत्रैव तीर्थे कुरुकुलोद्वह विप्राणा...,व्रतोपनयनाभ्यां चाप्युपवासेन वाप्युत ॥,"anuṣṭubh (1,2: pathyā; 3,4: pathyā)",32,Anuṣṭubh,Anuṣṭubh,NaN,Objective:\n\nGenerate a sanskrit verse in Anu...,test,NaN,तपो व्रत च योगेन च विप्राणां व्रत विधिः । \nउ...
2,तैः षड्भिरयनं वर्षं द्वेऽयने दक्षिणोत्तरे अयनं...,छः महीनों का एक अयन और दक्षिणायन तथा उत्तरायण ...,"anuṣṭubh (1,2: pathyā; 3,4: pathyā)",32,Anuṣṭubh,Anuṣṭubh,NaN,Objective:\n\nGenerate a sanskrit verse in Anu...,test,NaN,छ चतुर्मास्यं युगं वर्षं दक्षिणायनमुत्तरायणौ च...
3,भूय एवाहमिच्छामि महर्षेस्तस्य धीमतः कर्मणां वि...,युधिष्ठिरने कहा— द्विजश्रेष्ठ ! मैं पुनः बुद्ध...,"anuṣṭubh (1,2: pathyā; 3,4: pathyā)",32,Anuṣṭubh,Anuṣṭubh,NaN,Objective:\n\nGenerate a sanskrit verse in Anu...,test,NaN,युधिष्ठिरो वचनं द्विजश्रेष्ठं बुद्धिमान्‍ महर्...
4,नैषां पश्यामि हन्तारं प्राणिनां संयुगे परम् वि...,इन प्राणशक्तिसम्पन्न नरेशोंको कोई दूसरा वीर सं...,"anuṣṭubh (1,2: pathyā; 3,4: pathyā)",32,Anuṣṭubh,Anuṣṭubh,NaN,Objective:\n\nGenerate a sanskrit verse in Anu...,test,NaN,न दृष्टं मे वीरैर्युद्धे नरेशानां शक्तिसंपन्ना...


In [26]:
mask = df["model_out"].str.contains(
    r"sansk\s*text\s*=",
    case=False,
    na=False,
    regex=True
)

mask.sum()

np.int64(0)

In [27]:
import re

mask = df["model_out"].fillna("").str.contains(
    r"[^\u0900-\u097F\s।॥]",
    regex=True
)

mask.sum()

np.int64(722)

In [28]:
problem_rows = df[df["model_out"].fillna("").str.contains(r"[A-Za-z]", regex=True)]

problem_rows.to_csv("temp.csv", index=False)

In [29]:
def clean_output(text):
    if not isinstance(text, str):
        return text

    # Remove Markdown code fences
    text = re.sub(r'^```[^\n]*\n?', '', text)
    text = re.sub(r'\n?```$', '', text)

    # Keep everything before the first English/commentary line
    text = re.split(
        r'\n\s*[\(\[\*"\'\-]*\s*[A-Za-zĀāĪīŪūṚṛṜṝḶḷḸḹṂṃṀṁṄṅÑñṬṭḌḍṆṇŚśṢṣḤḥ]',
        text,
        maxsplit=1
    )[0]

    return text.strip()

df["filtered_model_out"] = df["model_out"].apply(clean_output)

In [30]:
problem_rows = df[df["filtered_model_out"].fillna("").str.contains(r"[A-Za-z]", regex=True)]

print(problem_rows.shape)
problem_rows.to_csv("temp.csv", index=False)

(0, 12)


In [31]:
import sys
from pathlib import Path

sys.path.append(str(Path("chandas-detector").resolve()))

import chandas_detector
from chandas_detector import detect_meter, format_result

In [32]:
meter_codes = []

for r in tqdm(df.itertuples(index=False), total=len(df)):
    verse = r.filtered_model_out.strip()
    result = detect_meter(verse)

    meter_codes.append(result.meter if result.confidence == "exact" else None)

df["out_meter"] = meter_codes

  0%|          | 0/2919 [00:00<?, ?it/s]

100%|██████████| 2919/2919 [00:01<00:00, 1906.65it/s]


In [33]:
df.head()

,sa,hi,skr_out,syllable_count,skr_meter,meter_cd,comments,prompt,split,thinking,model_out,filtered_model_out,out_meter
0,उक्तोऽयमर्थ आचार्यों मया कर्णेन चासकृत् पुनरेव...,आचार्यो ! मैंने और कर्णने यह बात आपलोगों से कई...,"anuṣṭubh (1,2: pathyā; 3,4: pathyā)",32,Anuṣṭubh,Anuṣṭubh,NaN,Objective:\n\nGenerate a sanskrit verse in Anu...,test,NaN,आचार्य तव वचनं बहु विनिपातं विनिपातं च विनिपात...,आचार्य तव वचनं बहु विनिपातं विनिपातं च विनिपात...,None
1,अर्धकीलं च तत्रैव तीर्थे कुरुकुलोद्वह विप्राणा...,व्रतोपनयनाभ्यां चाप्युपवासेन वाप्युत ॥,"anuṣṭubh (1,2: pathyā; 3,4: pathyā)",32,Anuṣṭubh,Anuṣṭubh,NaN,Objective:\n\nGenerate a sanskrit verse in Anu...,test,NaN,तपो व्रत च योगेन च विप्राणां व्रत विधिः । \nउ...,तपो व्रत च योगेन च विप्राणां व्रत विधिः । \nउ...,None
2,तैः षड्भिरयनं वर्षं द्वेऽयने दक्षिणोत्तरे अयनं...,छः महीनों का एक अयन और दक्षिणायन तथा उत्तरायण ...,"anuṣṭubh (1,2: pathyā; 3,4: pathyā)",32,Anuṣṭubh,Anuṣṭubh,NaN,Objective:\n\nGenerate a sanskrit verse in Anu...,test,NaN,छ चतुर्मास्यं युगं वर्षं दक्षिणायनमुत्तरायणौ च...,छ चतुर्मास्यं युगं वर्षं दक्षिणायनमुत्तरायणौ च...,None
3,भूय एवाहमिच्छामि महर्षेस्तस्य धीमतः कर्मणां वि...,युधिष्ठिरने कहा— द्विजश्रेष्ठ ! मैं पुनः बुद्ध...,"anuṣṭubh (1,2: pathyā; 3,4: pathyā)",32,Anuṣṭubh,Anuṣṭubh,NaN,Objective:\n\nGenerate a sanskrit verse in Anu...,test,NaN,युधिष्ठिरो वचनं द्विजश्रेष्ठं बुद्धिमान्‍ महर्...,युधिष्ठिरो वचनं द्विजश्रेष्ठं बुद्धिमान्‍ महर्...,None
4,नैषां पश्यामि हन्तारं प्राणिनां संयुगे परम् वि...,इन प्राणशक्तिसम्पन्न नरेशोंको कोई दूसरा वीर सं...,"anuṣṭubh (1,2: pathyā; 3,4: pathyā)",32,Anuṣṭubh,Anuṣṭubh,NaN,Objective:\n\nGenerate a sanskrit verse in Anu...,test,NaN,न दृष्टं मे वीरैर्युद्धे नरेशानां शक्तिसंपन्ना...,न दृष्टं मे वीरैर्युद्धे नरेशानां शक्तिसंपन्ना...,None


In [34]:
(df["meter_cd"] == df["out_meter"]).sum()/df.shape[0]*100

np.float64(0.0)

In [35]:
df.isnull().sum()

sa                       0
hi                       0
skr_out                  0
syllable_count           0
skr_meter                0
meter_cd                 0
comments              2919
prompt                   0
split                    0
thinking              2919
model_out                0
filtered_model_out       0
out_meter             2919
dtype: int64

In [24]:
df[~df['out_meter'].isnull()]

,sa,hi,skr_out,syllable_count,skr_meter,meter_cd,comments,prompt,split,model_out,filtered_model_out,out_meter
363,स एव खलु दारुभ्यो यदा निर्मथ्य दीप्यते तद्दारु...,वही अनि यदि काष्ठ से मथकर उद्दीप्त कर दी जाती ...,"anuṣṭubh (1,2: pathyā; 3,4: pathyā)",32,Anuṣṭubh,Anuṣṭubh,NaN,Objective:\n\nGenerate a sanskrit verse in Anu...,test,अग्निः काष्ठात् प्रदीप्तः सन् \nवह्निर्वह्निं...,अग्निः काष्ठात् प्रदीप्तः सन् \nवह्निर्वह्निं...,Anuṣṭubh
702,त्याज्यं दोषवदित्येके कर्म प्राहुर्मनीषिणः यज्...,तथा कई एक विद्वान् ऐसा कहते हैं कि कर्म सभी दो...,"anuṣṭubh (1,2: pathyā; 3,4: pathyā)",32,Anuṣṭubh,Anuṣṭubh,NaN,Objective:\n\nGenerate a sanskrit verse in Anu...,test,कर्माणि दोषयुक्तानि त्याज्यानि वदते बुधाः । \...,कर्माणि दोषयुक्तानि त्याज्यानि वदते बुधाः । \...,Anuṣṭubh
2013,अहमात्मा गुडाकेश सर्वभूताशयस्थितः अहमादिश्च मध...,हे अर्जुन! मैं सब भूतोंके हृदयमें स्थित सबका आ...,"anuṣṭubh (1,2: pathyā; 3,4: pathyā)",32,Anuṣṭubh,Anuṣṭubh,NaN,Objective:\n\nGenerate a sanskrit verse in Anu...,test,अर्जुन! सर्वभूतानां हृदयस्थोऽहमात्मना। \nसर्व...,अर्जुन! सर्वभूतानां हृदयस्थोऽहमात्मना। \nसर्व...,Anuṣṭubh
2157,प्रवादेनेह मत्स्यानां राजा नाम्नायमुच्यते अहमे...,राजा विराट तो कहने के लिये ही मत्स्यदेशका नामम...,"anuṣṭubh (1,2: pathyā; 3,4: pathyā)",32,Anuṣṭubh,Anuṣṭubh,NaN,Objective:\n\nGenerate a sanskrit verse in Anu...,test,विराटो नाममात्रं राजा मत्स्यानां वदन्ति ते। \...,विराटो नाममात्रं राजा मत्स्यानां वदन्ति ते। \...,Anuṣṭubh
2727,श्रोत्रं त्वक् चैव चक्षुश्च जिह्वा घ्राणं च पञ...,"श्रोत्र, त्वचा, नेत्र, जिह्वा और पाँचवीं नासिक...","anuṣṭubh (1,2: pathyā; 3,4: pathyā)",32,Anuṣṭubh,Anuṣṭubh,NaN,Objective:\n\nGenerate a sanskrit verse in Anu...,test,श्रोत्रं त्वचा च नेत्रं च जिह्वा नासिकया सह। ...,श्रोत्रं त्वचा च नेत्रं च जिह्वा नासिकया सह। ...,Anuṣṭubh
2821,ततस्तूर्यप्रणादश्च भेरीणां च महास्वना: जज्षिरि...,"तदनन्तर तुरही और भेरियाँ बज उठीं, उनकी आवाज बह...","anuṣṭubh (1,2: pathyā; 3,4: asamīcīnā, na prat...",32,Anuṣṭubh,Anuṣṭubh,NaN,Objective:\n\nGenerate a sanskrit verse in Anu...,test,तुरही भेरिणः शब्दं निनादं विस्तृतं गतम्। \nसि...,तुरही भेरिणः शब्दं निनादं विस्तृतं गतम्। \nसि...,Anuṣṭubh
